In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../assignments/DATA_OFFER_ALLOCATION.csv")

In [ ]:
print(df.columns)
print(len(df))

In [ ]:
df.head()

### EDA - Data Leakage

In [ ]:
print(f"Num Sessions: {len(df['SESSION_KEY'].unique())}")
print(f"Num Members: {len(df['MEMBER_KEY'].unique())}") # some users have several session
print(f"Min Session Date: {df['SESSION_DATE'].min()}; Max Session Date: {df['SESSION_DATE'].max()}")

### EDA - Univariable Distributions

In [ ]:
for col in df.columns.drop(['SESSION_KEY', 'SESSION_DATE', 'MEMBER_KEY']):
    print(df[col].value_counts(normalize=True))

### EDA - Data Quality

In [ ]:
valid_transaction_mask = (df['FLAG_TRANSACTION'] == 1) & (df['POINTS_PURCHASED'] > 0) & (df['PRICE_PER_POINT'] > 0) & (df['OFFER_RICHNESS_APPLIED'] != -1)
has_transaction_mask = df['FLAG_TRANSACTION'] == 1

# print(f"{len(df[valid_transaction_mask])}")
# print(f"{len(df[has_transaction_mask])}")
assert(len(df[has_transaction_mask]) == len(df[valid_transaction_mask]))

In [ ]:
valid_no_transaction_mask = (df['FLAG_TRANSACTION'] == 0) & (df['POINTS_PURCHASED'] == 0) & (df['PRICE_PER_POINT'] == 0) & (df['OFFER_RICHNESS_APPLIED'] == -1)
no_transaction_mask = df['FLAG_TRANSACTION'] == 0

# assert(len(df[valid_no_transaction_mask]) == len(df[no_transaction_mask]))


### EDA - 

In [ ]:
has_3k_minimum_balance_mask = df['CURRENT_BALANCE'] > 3000

print(f"Sessions with at least 3k in current balance: {len(df[has_3k_minimum_balance_mask]) / len(df) * 100}%")

In [ ]:
df[~has_3k_minimum_balance_mask]

In [ ]:
# points purchased if no offer
mask = (df['OFFER_RICHNESS_SERVED'] == -1) & (df['POINTS_PURCHASED'] > 0)
df[mask]

In [ ]:
# 
mask = (df['POINTS_PURCHASED'] > 0)
df[mask]

In [ ]:
# likelihood to convert based on: 
# - CURRENT_BALANCE, DAYS_SINCE_LAST_PURCHASE, DAYS_SINCE_LAST_VISIT_NO_PURCHASE


In [ ]:
mask_first_time_buyer = df['FLAG_FIRST_TIME_BUYER'] == 1
mask_first_time_visitor = df['FLAG_FIRST_TIME_VISITOR'] == 1
mask_points_purchased = df['POINTS_PURCHASED'] > 0
mask_positive_balance = df['CURRENT_BALANCE'] > 0
mask_positive_transaction_l12m = df['COUNT_TRANX_L12M'] > 0

# df[mask_first_time_buyer]['POINTS_PURCHASED']
print(len(df[mask_first_time_buyer & mask_points_purchased]) / len(df[mask_first_time_buyer]))
print(len(df[mask_first_time_visitor & mask_points_purchased]) / len(df[mask_first_time_visitor]))
print(len(df[mask_points_purchased]) / len(df))
print(len(df[mask_positive_balance & mask_points_purchased]) / len(df[mask_positive_balance]))

In [ ]:
df[mask_points_purchased]['POINTS_PURCHASED'].hist()

In [ ]:
df[mask_points_purchased]['POINTS_PURCHASED'].quantile(0.85) # 20k

In [ ]:
import numpy as np

np.log(df[mask_points_purchased]['POINTS_PURCHASED']).hist()
# np.log(np.log(df[mask_points_purchased]['POINTS_PURCHASED'])).hist()
# np.log(np.log(np.log(df[mask_points_purchased]['POINTS_PURCHASED']))).hist()

In [ ]:
data = df['CURRENT_BALANCE']
lower = np.percentile(data, 1)
upper = np.percentile(data, 95)
capped = np.clip(data, lower, upper)

# plt.hist(capped, bins=10)
# plt.show()

bins = pd.cut(capped, bins=10)
df_grouped = pd.DataFrame({
    'bucket': bins,
    'converted': mask_points_purchased
})


result = df_grouped.groupby('bucket')['converted'].agg(
    total='count',
    conversions='sum',
    conversion_rate='mean'
).sort_index()
print(result)

In [ ]:
# counts = pd.qcut(df['CURRENT_BALANCE'], q=4).value_counts().sort_index()
# counts.plot(kind='barh')

In [ ]:
# pd.cut(df['CURRENT_BALANCE'], bins=3).value_counts() 
# df['CURRENT_BALANCE'].max()




In [ ]:
# BUCKET_SIZE = 10
# JUMPS = (df['CURRENT_BALANCE'].max() - df['CURRENT_BALANCE'].min()) / BUCKET_SIZE
# print(JUMPS)
# pd.cut(df['CURRENT_BALANCE'], bins=[i*JUMPS for i in range(BUCKET_SIZE)]).value_counts().sort_index()

In [ ]:
for col in ['POINTS_PURCHASED', 'CURRENT_BALANCE', 'COUNT_TRANX_L12M']:
    data = df[col]
    lower = np.percentile(data, 1)
    upper = np.percentile(data, 99)
    capped = np.clip(data, lower, upper)

    plt.hist(capped, bins=10)
    plt.title(f"Histogram - {col}")
    plt.show()

In [ ]:
# df.groupby('COUNT_TRANX_L12M')['POINTS_PURCHASED'].count()
plt.scatter(df[mask_positive_transaction_l12m]['COUNT_TRANX_L12M'], df[mask_positive_transaction_l12m]['POINTS_PURCHASED'])


In [ ]:
data = df[df['DAYS_SINCE_LAST_VISIT_NO_PURCHASE'] < 9999]['DAYS_SINCE_LAST_VISIT_NO_PURCHASE']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original
axes[0].hist(data, bins=30)
axes[0].set_title('Original')
axes[0].set_xlabel('Days Since Last Visit No Purchase')
axes[0].set_ylabel('Frequency')

# Square root transformed
axes[1].hist(np.log(data), bins=30)
axes[1].set_title('Square Root Transformed')
axes[1].set_xlabel('Log(Days Since Last Visit No Purchase)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
data = df[df['DAYS_SINCE_LAST_PURCHASE_L12M'] < 9999]['DAYS_SINCE_LAST_PURCHASE_L12M']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original
axes[0].hist(data, bins=30)
axes[0].set_title('Original')
axes[0].set_xlabel('Days Since Last Purchase')
axes[0].set_ylabel('Frequency')

# Square root transformed
axes[1].hist(np.sqrt(data), bins=30)
axes[1].set_title('Square Root Transformed')
axes[1].set_xlabel('Sqrt(Days Since Last Purchase)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### EDA - Can we categorize continuous variable together?

In [ ]:
df.columns


In [ ]:
# EDA for users who performed transaction

mask_first_time_buyer = df['FLAG_FIRST_TIME_BUYER'] == 1
mask_first_time_visitor = df['FLAG_FIRST_TIME_VISITOR'] == 1
mask_points_purchased = df['POINTS_PURCHASED'] > 0





### EDA - Are users less happy if the offer they see is less than the one they saw before?

Compare `OFFER_RICHNESS_SERVED` (current session) vs `LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M` (offer at last purchase).  
Only applies to returning buyers (last offer != -1).  
"Less happy" = lower conversion rate and/or fewer points purchased.

In [ ]:

# Filter to returning buyers only (have a prior purchase in L12M)
mask_returning = df['LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M'] != -1
df_returning = df[mask_returning].copy()

# Offer delta: positive = better offer now, negative = worse offer now
df_returning['offer_delta'] = df_returning['OFFER_RICHNESS_SERVED'] - df_returning['LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M']

def offer_direction(delta):
    if delta < 0:
        return 'Worse (degraded)'
    elif delta == 0:
        return 'Same'
    else:
        return 'Better (improved)'

df_returning['offer_direction'] = df_returning['offer_delta'].apply(offer_direction)
order = ['Worse (degraded)', 'Same', 'Better (improved)']

# --- Conversion rate by offer direction ---
conv_by_direction = df_returning.groupby('offer_direction').agg(
    total=('FLAG_TRANSACTION', 'count'),
    conversions=('FLAG_TRANSACTION', 'sum'),
).assign(conversion_rate=lambda x: x['conversions'] / x['total']).loc[order]

print("Conversion rate by offer direction vs. last purchase offer:")
print(conv_by_direction.to_string())
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(conv_by_direction.index, conv_by_direction['conversion_rate'], color=['#d9534f', '#5bc0de', '#5cb85c'])
axes[0].set_title('Conversion Rate by Offer Direction')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_xlabel('Offer vs. Last Purchase Offer')
for i, v in enumerate(conv_by_direction['conversion_rate']):
    axes[0].text(i, v + 0.002, f'{v:.1%}', ha='center', fontsize=10)

# --- Points purchased distribution among converters ---
df_ret_buyers = df_returning[df_returning['FLAG_TRANSACTION'] == 1]
for direction, grp in df_ret_buyers.groupby('offer_direction'):
    axes[1].hist(np.log(grp['POINTS_PURCHASED'] + 1), bins=20, alpha=0.6, label=direction)

axes[1].set_title('log(Points Purchased) Among Converters\nby Offer Direction')
axes[1].set_xlabel('log(Points Purchased)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

# --- Mean points purchased by offer direction ---
mean_pts = df_ret_buyers.groupby('offer_direction')['POINTS_PURCHASED'].agg(['mean', 'median', 'count']).loc[order]
print("\nPoints Purchased stats (converters only):")
print(mean_pts.to_string())


### EDA - Do users buy more or fewer points than their last transaction? (Stratified)

Compare `POINTS_PURCHASED` (this session) vs `POINTS_PURCHASED_LAST_TRANX_L12M`.  
Restricted to current buyers who also have a prior transaction.  
Stratify by: offer direction (better/same/worse), offer richness tier, and transaction frequency.

In [ ]:

# Restrict to current buyers with a prior transaction (so we can compute a delta)
mask_repeat_buyer = (df['FLAG_TRANSACTION'] == 1) & (df['POINTS_PURCHASED_LAST_TRANX_L12M'] > 0)
df_repeat = df[mask_repeat_buyer].copy()

# Points delta vs last transaction
df_repeat['pts_delta'] = df_repeat['POINTS_PURCHASED'] - df_repeat['POINTS_PURCHASED_LAST_TRANX_L12M']
df_repeat['pts_ratio'] = df_repeat['POINTS_PURCHASED'] / df_repeat['POINTS_PURCHASED_LAST_TRANX_L12M']

def pts_direction(ratio):
    if ratio < 0.95:
        return 'Bought Less'
    elif ratio <= 1.05:
        return 'Bought Same (~)'
    else:
        return 'Bought More'

df_repeat['pts_direction'] = df_repeat['pts_ratio'].apply(pts_direction)
pts_order = ['Bought Less', 'Bought Same (~)', 'Bought More']

# Add offer direction for stratification (recompute on df_repeat)
df_repeat['offer_delta'] = df_repeat['OFFER_RICHNESS_SERVED'] - df_repeat['LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M']
df_repeat['offer_direction'] = df_repeat['offer_delta'].apply(
    lambda d: 'Worse' if d < 0 else ('Same' if d == 0 else 'Better')
)

# --- Overall distribution ---
overall = df_repeat['pts_direction'].value_counts().reindex(pts_order)
print("Overall: points bought vs. last transaction")
print(overall.to_string())
print(f"\nMedian ratio: {df_repeat['pts_ratio'].median():.2f}  |  Mean ratio: {df_repeat['pts_ratio'].mean():.2f}")
print()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Overall bar
axes[0, 0].bar(pts_order, overall.values, color=['#d9534f', '#5bc0de', '#5cb85c'])
axes[0, 0].set_title('Points Bought vs. Last Transaction (Overall)')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(overall.values):
    axes[0, 0].text(i, v + 1, str(v), ha='center')

# Plot 2: Stratified by offer direction
strat_offer = df_repeat.groupby(['offer_direction', 'pts_direction']).size().unstack(fill_value=0).reindex(
    columns=pts_order
)
strat_offer_pct = strat_offer.div(strat_offer.sum(axis=1), axis=0) * 100
strat_offer_pct.plot(kind='bar', ax=axes[0, 1], color=['#d9534f', '#5bc0de', '#5cb85c'], rot=0)
axes[0, 1].set_title('Points Direction by Offer Direction\n(% within group)')
axes[0, 1].set_ylabel('% of converters')
axes[0, 1].legend(title='Points Direction', loc='upper right', fontsize=8)

# Plot 3: Stratified by OFFER_RICHNESS_SERVED tier
strat_richness = df_repeat.groupby(['OFFER_RICHNESS_SERVED', 'pts_direction']).size().unstack(fill_value=0).reindex(
    columns=pts_order
)
strat_richness_pct = strat_richness.div(strat_richness.sum(axis=1), axis=0) * 100
strat_richness_pct.plot(kind='bar', ax=axes[1, 0], color=['#d9534f', '#5bc0de', '#5cb85c'], rot=0)
axes[1, 0].set_title('Points Direction by Offer Richness Served\n(% within group)')
axes[1, 0].set_ylabel('% of converters')
axes[1, 0].legend(title='Points Direction', fontsize=8)

# Plot 4: Stratified by transaction frequency bucket
df_repeat['tranx_bucket'] = pd.cut(
    df_repeat['COUNT_TRANX_L12M'], bins=[0, 1, 3, 6, 100], labels=['1 tx', '2-3 tx', '4-6 tx', '7+ tx'], right=True
)
strat_freq = df_repeat.groupby(['tranx_bucket', 'pts_direction'], observed=True).size().unstack(fill_value=0).reindex(
    columns=pts_order
)
strat_freq_pct = strat_freq.div(strat_freq.sum(axis=1), axis=0) * 100
strat_freq_pct.plot(kind='bar', ax=axes[1, 1], color=['#d9534f', '#5bc0de', '#5cb85c'], rot=0)
axes[1, 1].set_title('Points Direction by Transaction Frequency (L12M)\n(% within group)')
axes[1, 1].set_ylabel('% of converters')
axes[1, 1].legend(title='Points Direction', fontsize=8)

plt.tight_layout()
plt.show()

# --- Summary table: mean/median pts ratio by offer direction ---
print("\nMean pts ratio by offer direction (current offer vs. last purchase offer):")
print(df_repeat.groupby('offer_direction')['pts_ratio'].agg(['mean', 'median', 'count']).to_string())


In [ ]:
# Q7 — Member feature distributions by offer tier (selection bias check)
# Systematic differences in balance/activity across tiers => members are NOT randomly assigned.

features = {
    'CURRENT_BALANCE':             None,   # no sentinel, cap at p99
    'COUNT_TRANX_L12M':            None,
    'DAYS_SINCE_LAST_PURCHASE_L12M': 9999, # exclude sentinel
}

tiers = sorted(df['OFFER_RICHNESS_SERVED'].unique())
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (col, sentinel) in zip(axes, features.items()):
    data_by_tier, labels = [], []
    for tier in tiers:
        grp = df[df['OFFER_RICHNESS_SERVED'] == tier][col]
        if sentinel is not None:
            grp = grp[grp != sentinel]
        cap = np.percentile(grp, 99)
        data_by_tier.append(np.clip(grp, None, cap).values)
        labels.append(f'{tier:.0%}')
    ax.boxplot(data_by_tier, labels=labels, patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(col.replace('_', '\n'), fontsize=9)
    ax.set_xlabel('Offer Tier')
    ax.set_ylabel('Value (capped at p99)')

fig.suptitle('Q7 — Member Feature Distributions by Offer Tier\n(selection bias check)', y=1.02)
plt.tight_layout()
plt.show()

# Median summary
print("Median values by offer tier:")
for col, sentinel in features.items():
    sub = df if sentinel is None else df[df[col] != sentinel]
    print(sub.groupby('OFFER_RICHNESS_SERVED')[col].median().rename(col).to_frame().T.to_string())


In [ ]:
# Q8 — Raw conversion rate by offer tier; then controlled for member quality
# If the gap narrows when restricted to returning buyers => selection bias, not offer effect.

# Naive rates (all sessions)
naive = df.groupby('OFFER_RICHNESS_SERVED').agg(
    sessions=('FLAG_TRANSACTION', 'count'),
    conversions=('FLAG_TRANSACTION', 'sum'),
).assign(naive_rate=lambda x: x['conversions'] / x['sessions'])
print("Naive conversion rates:\n", naive.to_string(), "\n")

# Within returning buyers only
df_q8_ret = df[df['COUNT_TRANX_L12M'] > 0]
ret = df_q8_ret.groupby('OFFER_RICHNESS_SERVED').agg(
    sessions=('FLAG_TRANSACTION', 'count'),
    conversions=('FLAG_TRANSACTION', 'sum'),
).assign(returning_rate=lambda x: x['conversions'] / x['sessions'])
print("Returning-buyer conversion rates:\n", ret.to_string())

gap_naive = naive['naive_rate'].max() - naive['naive_rate'].min()
gap_ret   = ret['returning_rate'].max() - ret['returning_rate'].min()
print(f"\nNaive gap (max-min): {gap_naive:.3f}  |  Returning-buyer gap: {gap_ret:.3f}")
print("→ Gap narrows by", f"{(gap_naive - gap_ret)/gap_naive:.1%}", "when controlling for member quality")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(naive))
w = 0.35
ax.bar(x - w/2, naive['naive_rate'],   w, label='All sessions (naive)',  color='steelblue')
ax.bar(x + w/2, ret['returning_rate'], w, label='Returning buyers only', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels([f'{t:.0%}' for t in naive.index])
ax.set_xlabel('Offer Richness Tier')
ax.set_ylabel('Conversion Rate')
ax.set_title('Q8 — Conversion Rate by Offer Tier\nNaive vs. Returning Buyers (member quality control)')
ax.legend()
for i, (nv, rv) in enumerate(zip(naive['naive_rate'], ret['returning_rate'])):
    ax.text(i - w/2, nv + 0.003, f'{nv:.1%}', ha='center', fontsize=9)
    ax.text(i + w/2, rv + 0.003, f'{rv:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Q9 — Expected revenue per session (including zeros) by offer tier
# revenue = FLAG_TRANSACTION × POINTS_PURCHASED × PRICE_PER_POINT

df_q9 = df.copy()
df_q9['revenue'] = (df_q9['FLAG_TRANSACTION']
                    * df_q9['POINTS_PURCHASED']
                    * df_q9['PRICE_PER_POINT'].clip(lower=0))

rev_by_tier = df_q9.groupby('OFFER_RICHNESS_SERVED').agg(
    sessions=('revenue', 'count'),
    total_revenue=('revenue', 'sum'),
    mean_rev_per_session=('revenue', 'mean'),
    median_rev_per_session=('revenue', 'median'),
    conversion_rate=('FLAG_TRANSACTION', 'mean'),
)
print("Q9 — Expected revenue per session by offer tier:")
print(rev_by_tier.to_string())

fig, ax = plt.subplots(figsize=(7, 5))
tiers = [f'{t:.0%}' for t in rev_by_tier.index]
bars  = ax.bar(tiers, rev_by_tier['mean_rev_per_session'],
               color=['#e74c3c', '#3498db', '#2ecc71'])
ax.set_title('Q9 — Expected Revenue per Session (incl. zeros)\nby Offer Tier')
ax.set_xlabel('Offer Richness Tier')
ax.set_ylabel('Mean Revenue per Session ($)')
for bar, v in zip(bars, rev_by_tier['mean_rev_per_session']):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.05, f'${v:.2f}', ha='center')
plt.tight_layout()
plt.show()


In [ ]:
# Q10 — CURRENT_BALANCE vs POINTS_PURCHASED
# Do low-balance members top up to round-number milestones?

df_q10 = df[df['FLAG_TRANSACTION'] == 1].copy()
df_q10['post_balance'] = df_q10['CURRENT_BALANCE'] + df_q10['POINTS_PURCHASED']

balance_cap  = np.percentile(df_q10['CURRENT_BALANCE'], 99)
df_q10_plot  = df_q10[df_q10['CURRENT_BALANCE'] <= balance_cap]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_q10_plot['CURRENT_BALANCE'], df_q10_plot['POINTS_PURCHASED'],
                alpha=0.3, s=15, c='steelblue')
axes[0].set_xlabel('Current Balance (points)')
axes[0].set_ylabel('Points Purchased')
axes[0].set_title('Q10 — Balance vs. Points Purchased\n(transacting sessions, balance capped at p99)')

milestones  = [5_000, 10_000, 15_000, 20_000, 25_000, 30_000, 40_000, 50_000]
tolerance   = 500
hits = {m: ((df_q10['post_balance'] >= m - tolerance) & (df_q10['post_balance'] <= m + tolerance)).sum()
        for m in milestones}
ms_df = pd.Series(hits)
axes[1].bar([f'{m//1000}k' for m in ms_df.index], ms_df.values, color='steelblue')
axes[1].set_title(f'Q10 — Sessions Where post-purchase balance ≈ round milestone\n(±{tolerance} pts)')
axes[1].set_xlabel('Milestone')
axes[1].set_ylabel('Count of Sessions')

plt.tight_layout()
plt.show()

low_bal = df_q10[df_q10['CURRENT_BALANCE'] < 5000]
print(f"Low-balance buyers (<5k pts, n={len(low_bal)}) — median purchase: {low_bal['POINTS_PURCHASED'].median():.0f}")
print("Top purchase sizes:", low_bal['POINTS_PURCHASED'].value_counts().head(5).to_dict())


In [ ]:
# Q12 — Revenue concentration: Lorenz curve
# How many sessions account for 80% of total revenue?

df_q12 = df.copy()
df_q12['revenue'] = (df_q12['FLAG_TRANSACTION']
                     * df_q12['POINTS_PURCHASED']
                     * df_q12['PRICE_PER_POINT'].clip(lower=0))

rev_sorted  = df_q12['revenue'].sort_values().reset_index(drop=True)
cum_rev     = rev_sorted.cumsum() / rev_sorted.sum()
cum_sess    = np.arange(1, len(rev_sorted) + 1) / len(rev_sorted)

idx_80          = np.searchsorted(cum_rev.values, 0.80)
pct_sess_80     = cum_sess[idx_80] * 100
top10_rev_share = df_q12.nlargest(int(0.10 * len(df_q12)), 'revenue')['revenue'].sum() / df_q12['revenue'].sum()

print(f"Total revenue:  ${df_q12['revenue'].sum():,.0f}")
print(f"Top 10% sessions drive {top10_rev_share:.1%} of revenue")
print(f"Sessions needed for 80% of revenue: {idx_80:,} ({pct_sess_80:.1f}% of all sessions)")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(cum_sess * 100, cum_rev * 100, color='steelblue', lw=2, label='Lorenz curve')
ax.plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Perfect equality')
ax.axhline(80, color='red',    linestyle=':', alpha=0.7, label='80% revenue line')
ax.axvline(pct_sess_80, color='orange', linestyle=':', alpha=0.7,
           label=f'{pct_sess_80:.1f}% of sessions → 80% rev')
ax.set_xlabel('Cumulative % of Sessions (sorted by revenue asc)')
ax.set_ylabel('Cumulative % of Total Revenue')
ax.set_title('Q12 — Revenue Concentration (Lorenz Curve)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Q13 — Conversion rate: first-time vs. returning buyers, within each offer tier
# If the gap persists within tiers => segmentation is genuinely informative.
# If it disappears within tiers => offer assignment was doing the work.

q13 = df.groupby(['OFFER_RICHNESS_SERVED', 'FLAG_FIRST_TIME_BUYER']).agg(
    sessions=('FLAG_TRANSACTION', 'count'),
    conversions=('FLAG_TRANSACTION', 'sum'),
).assign(conv_rate=lambda x: x['conversions'] / x['sessions']).reset_index()

q13['buyer_type'] = q13['FLAG_FIRST_TIME_BUYER'].map({0: 'Returning buyer', 1: 'First-time buyer'})
pivot = q13.pivot(index='OFFER_RICHNESS_SERVED', columns='buyer_type', values='conv_rate')
print("Q13 — Conversion rate by offer tier × buyer type:")
print(pivot.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(pivot))
w = 0.35
ax.bar(x - w/2, pivot['First-time buyer'], w, label='First-time buyer', color='#e74c3c')
ax.bar(x + w/2, pivot['Returning buyer'],  w, label='Returning buyer',  color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels([f'{t:.0%}' for t in pivot.index])
ax.set_xlabel('Offer Richness Tier')
ax.set_ylabel('Conversion Rate')
ax.set_title('Q13 — Conversion Rate by Offer Tier\nFirst-time vs. Returning Buyers')
ax.legend()
for i, (ftb, rb) in enumerate(zip(pivot['First-time buyer'], pivot['Returning buyer'])):
    ax.text(i - w/2, ftb + 0.003, f'{ftb:.1%}', ha='center', fontsize=8)
    ax.text(i + w/2, rb  + 0.003, f'{rb:.1%}',  ha='center', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Q14 — Recency vs. frequency: which predicts conversion better for returning buyers?
# Expected: recency shows a sharper cutoff around 90 days.

df_q14 = df[(df['DAYS_SINCE_LAST_PURCHASE_L12M'] != 9999) & (df['COUNT_TRANX_L12M'] > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recency buckets
recency_bins   = [0, 30, 60, 90, 120, 180, 365]
recency_labels = ['0-30d', '31-60d', '61-90d', '91-120d', '121-180d', '181-365d']
df_q14['recency_bucket'] = pd.cut(df_q14['DAYS_SINCE_LAST_PURCHASE_L12M'], bins=recency_bins, labels=recency_labels)
rec_conv = df_q14.groupby('recency_bucket', observed=True)['FLAG_TRANSACTION'].agg(['mean', 'count'])
axes[0].bar(rec_conv.index, rec_conv['mean'], color='steelblue')
axes[0].set_title('Q14 — Conversion Rate by Recency\n(returning buyers)')
axes[0].set_xlabel('Days Since Last Purchase')
axes[0].set_ylabel('Conversion Rate')
axes[0].tick_params(axis='x', rotation=30)
for i, (v, n) in enumerate(zip(rec_conv['mean'], rec_conv['count'])):
    axes[0].text(i, v + 0.003, f'{v:.1%}\n(n={n})', ha='center', fontsize=8)

# Frequency buckets
freq_bins   = [0, 1, 2, 4, 7, 100]
freq_labels = ['1', '2', '3-4', '5-7', '8+']
df_q14['freq_bucket'] = pd.cut(df_q14['COUNT_TRANX_L12M'], bins=freq_bins, labels=freq_labels)
freq_conv = df_q14.groupby('freq_bucket', observed=True)['FLAG_TRANSACTION'].agg(['mean', 'count'])
axes[1].bar(freq_conv.index, freq_conv['mean'], color='darkorange')
axes[1].set_title('Q14 — Conversion Rate by Frequency\n(returning buyers, L12M txns)')
axes[1].set_xlabel('Transactions in Last 12 Months')
axes[1].set_ylabel('Conversion Rate')
for i, (v, n) in enumerate(zip(freq_conv['mean'], freq_conv['count'])):
    axes[1].text(i, v + 0.003, f'{v:.1%}\n(n={n})', ha='center', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Q15 — L12M purchasers vs. non-purchasers: feature space comparison
# Justifies the has_l12m_purchase binary flag and sentinel encoding strategy.

df_q15 = df.copy()
df_q15['group'] = (df_q15['COUNT_TRANX_L12M'] > 0).map({True: 'Has L12M Purchase', False: 'No L12M Purchase'})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Balance distribution
for group, grp in df_q15.groupby('group'):
    cap = np.percentile(grp['CURRENT_BALANCE'], 99)
    axes[0, 0].hist(np.clip(grp['CURRENT_BALANCE'], None, cap), bins=30, alpha=0.6, label=group, density=True)
axes[0, 0].set_title('Current Balance Distribution')
axes[0, 0].set_xlabel('Current Balance (capped at p99)')
axes[0, 0].legend()

# 2. Offer tier received (% within group)
offer_dist = df_q15.groupby(['group', 'OFFER_RICHNESS_SERVED']).size().unstack(fill_value=0)
offer_dist.div(offer_dist.sum(axis=1), axis=0).mul(100).T.plot(kind='bar', ax=axes[0, 1], rot=0)
axes[0, 1].set_title('Offer Tier Received (% within group)')
axes[0, 1].set_xlabel('Offer Richness Tier')
axes[0, 1].set_ylabel('% of group')

# 3. Conversion rate
conv_comp = df_q15.groupby('group')['FLAG_TRANSACTION'].agg(['mean', 'count'])
axes[1, 0].bar(conv_comp.index, conv_comp['mean'], color=['#e74c3c', '#2ecc71'])
axes[1, 0].set_title('Conversion Rate')
axes[1, 0].set_ylabel('Conversion Rate')
for i, (v, n) in enumerate(zip(conv_comp['mean'], conv_comp['count'])):
    axes[1, 0].text(i, v + 0.003, f'{v:.1%}\n(n={n:,})', ha='center', fontsize=9)

# 4. Points purchased among buyers
df_q15_buyers = df_q15[df_q15['FLAG_TRANSACTION'] == 1]
pts_comp = df_q15_buyers.groupby('group')['POINTS_PURCHASED'].agg(['mean', 'median', 'count'])
x = np.arange(len(pts_comp))
axes[1, 1].bar(x - 0.2, pts_comp['mean'],   0.4, label='Mean',   color='steelblue')
axes[1, 1].bar(x + 0.2, pts_comp['median'], 0.4, label='Median', color='darkorange')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(pts_comp.index)
axes[1, 1].set_title('Points Purchased (converters only)')
axes[1, 1].set_ylabel('Points')
axes[1, 1].legend()

fig.suptitle('Q15 — L12M Purchasers vs. Non-Purchasers', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(df_q15.groupby('group').agg(
    n=('FLAG_TRANSACTION', 'count'),
    conv_rate=('FLAG_TRANSACTION', 'mean'),
    median_balance=('CURRENT_BALANCE', 'median'),
).to_string())


In [ ]:
# Q16 — Among converters, what share purchased exactly 3,000 points (the minimum)?
# High share => price-constrained members; poor targets for the 40% offer.

df_q16 = df[df['FLAG_TRANSACTION'] == 1].copy()
n_total = len(df_q16)
n_min   = (df_q16['POINTS_PURCHASED'] == 3000).sum()
print(f"Converters buying exactly 3,000 pts: {n_min} / {n_total} = {n_min/n_total:.1%}")

min_by_tier = df_q16.groupby('OFFER_RICHNESS_SERVED').apply(
    lambda x: pd.Series({
        'n_converters':    len(x),
        'n_min_purchase':  (x['POINTS_PURCHASED'] == 3000).sum(),
        'pct_min_purchase':(x['POINTS_PURCHASED'] == 3000).mean() * 100,
    })
)
min_by_buyer = df_q16.groupby('FLAG_FIRST_TIME_BUYER').apply(
    lambda x: pd.Series({
        'n_converters':    len(x),
        'pct_min_purchase':(x['POINTS_PURCHASED'] == 3000).mean() * 100,
    })
).rename(index={0: 'Returning', 1: 'First-time'})
print("\nBy offer tier:\n", min_by_tier.to_string())
print("\nBy buyer type:\n", min_by_buyer.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

top_sizes = df_q16['POINTS_PURCHASED'].value_counts().nlargest(15).sort_index()
colors = ['#e74c3c' if x == 3000 else 'steelblue' for x in top_sizes.index]
axes[0].bar([str(x) for x in top_sizes.index], top_sizes.values, color=colors)
axes[0].set_title('Top Purchase Sizes  (red = 3,000 minimum)')
axes[0].set_xlabel('Points Purchased')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=60)

axes[1].bar([f'{t:.0%}' for t in min_by_tier.index], min_by_tier['pct_min_purchase'],
            color=['#e74c3c', '#3498db', '#2ecc71'])
axes[1].set_title('Q16 — % Converters Buying Minimum (3,000 pts)\nby Offer Tier')
axes[1].set_xlabel('Offer Richness Tier')
axes[1].set_ylabel('% of Converters')
for i, v in enumerate(min_by_tier['pct_min_purchase']):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center')

plt.suptitle('Q16 — Price-Constrained Buyers', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

df_buyers = df[df['POINTS_PURCHASED'] > 0].copy()

# Cap balance at p99 to reduce outlier distortion
balance_cap = df_buyers['CURRENT_BALANCE'].quantile(0.99)
df_buyers = df_buyers[df_buyers['CURRENT_BALANCE'] <= balance_cap]
df_buyers['CURRENT_BALANCE'] = np.log1p(df_buyers['CURRENT_BALANCE'])

palette = {0.40: 'red', 0.45: 'blue', 0.50: 'green'}

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df_buyers,
    x='CURRENT_BALANCE',
    y='POINTS_PURCHASED',
    hue='OFFER_RICHNESS_SERVED',
    palette=palette,
    alpha=0.5,
    s=30,
    ax=ax,
)
ax.set_title('Current Balance vs Points Purchased by Offer Richness')
ax.set_xlabel('Current Balance (capped at p99)')
ax.set_ylabel('Points Purchased')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [f'Offer {l}' for l in labels], title='Offer Richness')
plt.tight_layout()
plt.show()